[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/E.%20Integrated%2C%20Resilient%2C%20%26%20Modern%20Supply%20Chains%20%28The%20Frontier%29/094.%20The%20Multi-Echelon%20Inventory%20Optimization%20Problem/P94-Tier-7_executed.ipynb)

---


# 94. The Multi-Echelon Inventory Optimization Problem

## Tier 7: Human–AI Partnership (The Centaur Planner)

### Purpose and scope

This notebook illustrates a human–AI decision workflow for inventory planning. A rule-based AI component calculates a recommended inventory level and explains the factors used. A human planner reviews that recommendation, may adjust it using contextual information, and records a decision. The workflow then simulates an outcome and collects collaboration metrics.

The notebook is an educational demonstration of the partnership process, not a validated multi-echelon inventory optimizer. It does not optimize inventory across multiple locations, estimate a demand distribution from data, or establish that collaboration outperforms AI-only or human-only decisions. Its service-level outcomes are generated by a simple random simulation and should not be interpreted as empirical evidence.

### Inputs and outputs

Each scenario provides a location, SKU, current inventory, forecast demand, demand volatility, supplier reliability, lead time, and current and target service levels. The workflow returns an AI recommendation, a human decision, a simulated outcome, and summary metrics for the decisions processed.

The scenario dictionary is the input record. The dataclasses defined in the code give the recommendation, decision, combined result, and summary metrics explicit fields, making the information passed between stages inspectable. The processing order is: scenario data → AI recommendation → human review → simulated outcome → feedback record and summary.

### Model assumptions

- Forecast demand and demand volatility are expressed in units per day; lead time is expressed in days. The safety-stock expression therefore scales daily volatility by the square root of lead time.
- Supplier reliability is represented by a number between 0 and 1. The recommendation heuristic increases the base-stock estimate by dividing by this value.
- The safety-stock factor is a simplified rule selected from two values according to the target service level. It is not a precise quantile calculation for every target.
- Human contextual rules are illustrative, predefined rules. They do not represent a trained human model or a complete record of real-world planning practice. When several rules apply, later rules in `review_ai_recommendation` can replace the current action, final level, and reasoning; the contextual-factor list can retain information from more than one applicable rule.
- The partnership processes each scenario with the first expert in `human_experts`. The other initialized experts are available in the list but are not assigned scenarios by this example.
- The simulated outcome is not calculated from a physical inventory trajectory. Its service level is sampled independently of the inventory level, so it cannot validate the recommendation's effectiveness. The cost calculation is also illustrative rather than a calibrated inventory-cost model.
- The feedback functions record insights, but the example does not implement parameter updates or a learning algorithm.

### Workflow

1. The AI component calculates a recommended level and attaches reasoning and supporting data.
2. A human expert reviews the recommendation against predefined contextual knowledge.
3. The human approves, modifies, or overrides the recommendation.
4. A demonstration outcome is simulated and stored with the decision.
5. The workflow summarizes decisions, outcomes, and simple collaboration indicators.

Each call to `process_inventory_decision` executes these stages in sequence. The collaboration history is the accumulated record used by the reporting functions; the workflow is therefore sequential across scenarios as well as within each scenario.

### Recommendation heuristic

The AI component uses a base-stock calculation with a safety-stock term:

$$
SS = z\,\sigma\sqrt{L}, \qquad B = \mu L + SS, \qquad R = \frac{B}{r}.
$$

Here, $SS$ is estimated safety stock, $z$ is the selected safety-stock factor, $\sigma$ is daily demand volatility, $L$ is lead time in days, $\mu$ is forecast daily demand, $B$ is the base-stock estimate, $r$ is supplier reliability, and $R$ is the recommended inventory level. In the code, these quantities correspond to `required_safety_stock`, `base_stock`, and `recommended_level`. The factor is set to $1.65$ when the target service level is at least $0.95$, and to $1.28$ otherwise. This two-value rule is deliberately a heuristic, not a general conversion from service level to a statistical quantile.

The recommendation action is selected by comparing current inventory $I$ with two thresholds around $R$:

$$
a(I,R) =
\begin{cases}
\text{decrease stock}, & I > 1.2R,\\
\text{increase stock}, & I < 0.8R,\\
\text{maintain}, & 0.8R \leq I \leq 1.2R.
\end{cases}
$$

The code implements these comparisons in `analyze_inventory_situation`. The recommendation level is a target, while the action is a categorical description of how current inventory compares with that target; the action itself does not calculate an order quantity.

This is a transparent heuristic, not an optimization problem with a cost-minimizing objective. The notebook's central method is the sequential human review and feedback workflow described above.

### Human review and simulated outcomes

The human review begins by accepting the AI recommendation. Predefined contextual rules may then change the action and final inventory level. The returned `HumanDecision` stores the selected action, final level, explanation, confidence, and contextual factors. These rules illustrate how contextual information can enter a decision; they are not learned from the scenario outcomes.

The outcome simulator starts from a baseline service level of $0.95$. Depending on the human action, it adds or subtracts a random value from the specified interval, then clips the result to $[0.85, 0.99]$. The resulting service level $s$ is independent of the inventory level. The illustrative cost calculation is

$$
C_{\text{holding}} = 2F, \qquad
C_{\text{stockout}} =
\begin{cases}
100, & s < 0.95,\\
0, & s \geq 0.95,
\end{cases}
\qquad
C_{\text{total}} = C_{\text{holding}} + C_{\text{stockout}},
$$

where $F$ is the human's final inventory level. The code also sets the customer-satisfaction index to $100s$. These equations describe the demonstration simulator, not a physical inventory process or an empirically estimated cost function.

### Collaboration indicators

For $n$ processed decisions, the approval rate is the fraction with action `approve`; the modification/override frequency is the fraction with action `modify` or `override`; and average confidence is the arithmetic mean of the human confidence values. These are descriptive quantities, not validated measures of trust, calibration, or decision quality.

The reported early-to-recent service-level indicator is calculated only when $n > 10$:

$$
G = \frac{\bar{s}_{\mathrm{recent}} - \bar{s}_{\mathrm{early}}}{\bar{s}_{\mathrm{early}}},
$$

where $\bar{s}_{\mathrm{early}}$ is the mean of the first five simulated service levels and $\bar{s}_{\mathrm{recent}}$ is the mean of the last five. With ten or fewer records the code reports zero. This relative difference is not evidence of learning: outcomes are stochastic, there is no learning update, and the simulator does not link service level causally to inventory decisions.

### Interpreting the results

An approval is counted as agreement with the AI recommendation; a modification or override is counted in the modification/override-frequency indicator. These are operational definitions for this demonstration, not comprehensive measures of trust. A small number of simulated decisions cannot establish a trend, prove improved decision quality, or calibrate confidence. Read the explanation and scenario context alongside the numerical metrics.

In [ ]:
import random
from dataclasses import dataclass, field
from datetime import datetime
from typing import Any, Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# A consistent visual style makes comparisons across the report easier to read.
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

In [ ]:
@dataclass
class AIRecommendation:
    """An explainable inventory recommendation produced by the heuristic."""

    recommendation_id: str
    timestamp: datetime
    location: str
    sku: str
    action_type: str
    current_level: float
    recommended_level: float
    confidence_score: float
    reasoning: List[str]
    supporting_data: Dict[str, float]
    risk_assessment: str


@dataclass
class HumanDecision:
    """A planner's decision and contextual explanation."""

    decision_id: str
    recommendation_id: str
    human_expert: str
    timestamp: datetime
    action: str
    final_level: Optional[float]
    reasoning: str
    confidence: float
    contextual_factors: List[str]


@dataclass
class CentaurResult:
    """The combined record of one human–AI decision."""

    recommendation_id: str
    ai_recommendation: AIRecommendation
    human_decision: HumanDecision
    final_action: str
    performance_outcome: Optional[Dict[str, float]] = None
    learning_insights: List[str] = field(default_factory=list)


@dataclass
class TrustMetrics:
    """Simple descriptive indicators for a set of collaboration decisions."""

    timestamp: datetime
    agreement_rate: float
    average_confidence: float
    override_frequency: float
    learning_rate: float

In [ ]:
class ExplainableAIEngine:
    """Rule-based inventory recommendation engine with feedback logging."""

    def __init__(self, model_version: str = "1.0"):
        self.model_version = model_version
        self.knowledge_base = self._initialize_knowledge_base()
        self.learning_history: List[CentaurResult] = []

    def _initialize_knowledge_base(self) -> Dict[str, Any]:
        """Set illustrative thresholds and cost assumptions used by the example."""
        return {
            "demand_patterns": {
                "seasonal_increase": 0.3,
                "trend_growth": 0.15,
                "volatility_threshold": 0.25,
            },
            "supplier_reliability": {
                "excellent": 0.95,
                "good": 0.85,
                "fair": 0.70,
                "poor": 0.50,
            },
            "service_level_targets": {
                "critical_sku": 0.99,
                "standard_sku": 0.95,
                "slow_moving": 0.90,
            },
            "cost_optimization": {
                "holding_cost_rate": 0.25,
                "stockout_cost_multiplier": 10.0,
                "reorder_cost": 50.0,
            },
        }

    def analyze_inventory_situation(self, location_data: Dict[str, Any]) -> AIRecommendation:
        """Apply the stated heuristic and explain the resulting recommendation."""
        required_fields = {
            "location",
            "sku",
            "current_inventory",
            "demand_forecast",
            "demand_volatility",
            "supplier_reliability",
            "lead_time",
            "current_service_level",
            "target_service_level",
        }
        missing_fields = required_fields.difference(location_data)
        if missing_fields:
            raise ValueError(f"Scenario is missing required fields: {sorted(missing_fields)}")

        current_inventory = float(location_data["current_inventory"])
        demand_forecast = float(location_data["demand_forecast"])
        demand_volatility = float(location_data["demand_volatility"])
        supplier_reliability = float(location_data["supplier_reliability"])
        lead_time = float(location_data["lead_time"])
        service_level = float(location_data["current_service_level"])
        target_service_level = float(location_data["target_service_level"])

        if current_inventory < 0 or demand_forecast < 0 or demand_volatility < 0 or lead_time < 0:
            raise ValueError("Inventory, demand, volatility, and lead time must be non-negative.")
        if not 0 < supplier_reliability <= 1:
            raise ValueError("Supplier reliability must be greater than 0 and at most 1.")
        if not 0 <= service_level <= 1 or not 0 <= target_service_level <= 1:
            raise ValueError("Service levels must lie between 0 and 1.")

        # The two-factor rule is a teaching heuristic, not an exact service-level quantile.
        safety_stock_factor = 1.65 if target_service_level >= 0.95 else 1.28
        required_safety_stock = safety_stock_factor * demand_volatility * np.sqrt(lead_time)
        base_stock = demand_forecast * lead_time + required_safety_stock
        recommended_level = base_stock / supplier_reliability

        reasoning: List[str] = []
        if current_inventory > recommended_level * 1.2:
            action_type = "decrease_stock"
            reasoning.append(
                f"Current inventory ({current_inventory:.0f}) exceeds the heuristic target ({recommended_level:.0f})."
            )
            reasoning.append("Excess inventory may increase holding costs without improving service.")
            confidence = 0.85
            risk_assessment = "low"
        elif current_inventory < recommended_level * 0.8:
            action_type = "increase_stock"
            reasoning.append(
                f"Current inventory ({current_inventory:.0f}) is below the heuristic target ({recommended_level:.0f})."
            )
            reasoning.append("Low inventory may increase stockout risk and reduce service.")
            confidence = 0.90
            risk_assessment = "high" if supplier_reliability < 0.8 else "medium"
        else:
            action_type = "maintain"
            reasoning.append("Current inventory is within the heuristic's acceptable range.")
            reasoning.append("No immediate change is suggested; continue monitoring demand.")
            confidence = 0.75
            risk_assessment = "low"

        supporting_data = {
            "demand_forecast": demand_forecast,
            "demand_volatility": demand_volatility,
            "supplier_reliability": supplier_reliability,
            "lead_time": lead_time,
            "current_service_level": service_level,
            "target_service_level": target_service_level,
            "optimal_level": recommended_level,
            "safety_stock_required": required_safety_stock,
        }

        if demand_volatility > self.knowledge_base["demand_patterns"]["volatility_threshold"]:
            reasoning.append(
                f"Demand volatility ({demand_volatility:.2f}) exceeds the illustrative threshold."
            )
        if supplier_reliability < 0.8:
            reasoning.append(
                f"Supplier reliability ({supplier_reliability:.2f}) is low; the heuristic increases the buffer."
            )
        if service_level < target_service_level:
            reasoning.append(
                f"Current service level ({service_level:.2%}) is below the target ({target_service_level:.2%})."
            )

        timestamp = datetime.now()
        return AIRecommendation(
            recommendation_id=f"AI_{timestamp.strftime('%Y%m%d_%H%M%S_%f')}",
            timestamp=timestamp,
            location=str(location_data["location"]),
            sku=str(location_data["sku"]),
            action_type=action_type,
            current_level=current_inventory,
            recommended_level=float(recommended_level),
            confidence_score=confidence,
            reasoning=reasoning,
            supporting_data=supporting_data,
            risk_assessment=risk_assessment,
        )

    def learn_from_feedback(self, centaur_result: CentaurResult) -> None:
        """Record feedback for review; this demonstration does not update model parameters."""
        insights: List[str] = []
        decision = centaur_result.human_decision

        if decision.action in {"override", "modify"}:
            insights.append(f"Human {decision.action} recorded for {centaur_result.ai_recommendation.action_type}.")
            if centaur_result.performance_outcome is not None:
                service_level = centaur_result.performance_outcome["service_level"]
                insights.append(
                    f"Simulated outcome service level was {service_level:.1%}; no causal conclusion is implied."
                )
        elif decision.action == "approve":
            insights.append("AI recommendation approved; approval is recorded as a feedback signal.")
        else:
            insights.append("Recommendation rejected; rejection is recorded as a feedback signal.")

        centaur_result.learning_insights = insights
        self.learning_history.append(centaur_result)

    def _update_knowledge_base(self, centaur_result: CentaurResult) -> None:
        """Extension point for a future, explicitly specified learning rule."""
        self.learning_history.append(centaur_result)

In [ ]:
class HumanExpert:
    """Illustrative planner who applies predefined contextual review rules."""

    def __init__(self, name: str, expertise_level: str = "senior"):
        self.name = name
        self.expertise_level = expertise_level
        self.contextual_knowledge = self._initialize_contextual_knowledge()
        self.decision_history: List[HumanDecision] = []

    def _initialize_contextual_knowledge(self) -> Dict[str, Any]:
        """Provide the contextual facts used by the example decision rules."""
        return {
            "market_intelligence": {
                "upcoming_promotions": ["Summer Sale", "Back to School"],
                "competitor_actions": "Price reductions expected",
                "economic_indicators": "Consumer confidence stable",
            },
            "supplier_intelligence": {
                "new_suppliers": {
                    "Supplier_A": "Coming online in 2 months",
                    "Supplier_B": "Pilot testing",
                },
                "supplier_issues": {
                    "Supplier_C": "Quality problems",
                    "Supplier_D": "Capacity constraints",
                },
                "seasonal_patterns": "Q4 always higher demand",
            },
            "operational_constraints": {
                "warehouse_capacity": "Limited space at DC-Phoenix",
                "transportation_issues": "Port congestion expected",
                "labor_constraints": "Staff shortages in peak season",
            },
            "strategic_considerations": {
                "customer_priorities": "Premium service for key accounts",
                "inventory_strategy": "Lean inventory with buffer for critical items",
                "cost_focus": "Reduce holding costs while maintaining service",
            },
        }

    def review_ai_recommendation(self, ai_rec: AIRecommendation) -> HumanDecision:
        """Review an AI recommendation and return the planner's decision."""
        contextual_factors: List[str] = []
        human_reasoning = ""
        final_action = "approve"
        final_level = ai_rec.recommended_level
        human_confidence = ai_rec.confidence_score
        location = ai_rec.location
        sku = ai_rec.sku

        if "Summer Sale" in self.contextual_knowledge["market_intelligence"]["upcoming_promotions"]:
            if ai_rec.action_type == "decrease_stock":
                contextual_factors.append("Upcoming Summer Sale may increase demand")
                human_reasoning = (
                    "The upcoming promotion may increase demand, so the proposed reduction is not accepted."
                )
                final_action = "override"
                final_level = ai_rec.current_level * 1.2
                human_confidence = 0.90

        for supplier, info in self.contextual_knowledge["supplier_intelligence"]["new_suppliers"].items():
            if "Coming online" in info and ai_rec.action_type == "increase_stock":
                contextual_factors.append(f"New supplier {supplier} coming online soon")
                human_reasoning = f"{info}; maintain current levels while the supply change is pending."
                final_action = "modify"
                final_level = ai_rec.current_level
                human_confidence = 0.85
                break

        if "DC-Phoenix" in location and ai_rec.action_type == "increase_stock":
            if "Limited space" in self.contextual_knowledge["operational_constraints"]["warehouse_capacity"]:
                contextual_factors.append("Warehouse capacity constraints at DC-Phoenix")
                human_reasoning = (
                    "Warehouse capacity is limited, so only half of the proposed inventory increase is accepted."
                )
                final_action = "modify"
                final_level = ai_rec.current_level + (
                    ai_rec.recommended_level - ai_rec.current_level
                ) * 0.5
                human_confidence = 0.80

        if "critical" in sku.lower() and ai_rec.action_type == "decrease_stock":
            contextual_factors.append("Critical SKU requires a higher service buffer")
            human_reasoning = "The SKU is critical, so the planner retains a buffer rather than reducing stock."
            final_action = "modify"
            final_level = ai_rec.recommended_level * 1.1
            human_confidence = 0.95

        if not contextual_factors:
            human_reasoning = "The recommendation aligns with the available contextual information."

        timestamp = datetime.now()
        decision = HumanDecision(
            decision_id=f"HU_{timestamp.strftime('%Y%m%d_%H%M%S_%f')}",
            recommendation_id=ai_rec.recommendation_id,
            human_expert=self.name,
            timestamp=timestamp,
            action=final_action,
            final_level=float(final_level),
            reasoning=human_reasoning,
            confidence=human_confidence,
            contextual_factors=contextual_factors,
        )
        self.decision_history.append(decision)
        return decision

    def update_knowledge(self, outcome_data: Dict[str, float]) -> None:
        """Provide an extension point for a defined human-learning process."""
        # The demonstration records outcomes but does not infer new contextual facts.
        return None

In [ ]:
class CentaurPartnership:
    """Coordinate recommendation, human review, simulation, and reporting."""

    def __init__(self, ai_engine: ExplainableAIEngine, human_experts: List[HumanExpert]):
        if not human_experts:
            raise ValueError("At least one human expert is required.")
        self.ai_engine = ai_engine
        self.human_experts = human_experts
        self.collaboration_history: List[CentaurResult] = []
        self.trust_metrics: List[TrustMetrics] = []

    def process_inventory_decision(self, location_data: Dict[str, Any]) -> CentaurResult:
        """Run one scenario through the human–AI workflow in sequence."""
        ai_recommendation = self.ai_engine.analyze_inventory_situation(location_data)
        expert = self.human_experts[0]
        human_decision = expert.review_ai_recommendation(ai_recommendation)

        result = CentaurResult(
            recommendation_id=ai_recommendation.recommendation_id,
            ai_recommendation=ai_recommendation,
            human_decision=human_decision,
            final_action=human_decision.action,
        )
        result.performance_outcome = self._simulate_decision_outcome(result)
        self.ai_engine.learn_from_feedback(result)
        expert.update_knowledge(result.performance_outcome)
        self.collaboration_history.append(result)
        return result

    def _simulate_decision_outcome(self, centaur_result: CentaurResult) -> Dict[str, float]:
        """Generate an illustrative random outcome, not an inventory forecast."""
        base_service_level = 0.95
        action = centaur_result.human_decision.action

        if action == "approve":
            service_level = base_service_level + random.uniform(-0.02, 0.03)
        elif action == "modify":
            service_level = base_service_level + random.uniform(0.01, 0.04)
        elif action == "override":
            service_level = base_service_level + random.uniform(0.02, 0.05)
        else:
            service_level = base_service_level - random.uniform(0.01, 0.03)

        service_level = max(0.85, min(0.99, service_level))
        final_level = centaur_result.human_decision.final_level
        if final_level is None:
            final_level = centaur_result.ai_recommendation.current_level

        holding_cost = final_level * 2.0
        stockout_penalty = 100.0 if service_level < 0.95 else 0.0
        return {
            "service_level": service_level,
            "holding_cost": holding_cost,
            "total_cost": holding_cost + stockout_penalty,
            "customer_satisfaction": service_level * 100,
        }

    def calculate_trust_metrics(self) -> TrustMetrics:
        """Calculate descriptive approval, confidence, override, and trend indicators."""
        if not self.collaboration_history:
            return TrustMetrics(
                timestamp=datetime.now(),
                agreement_rate=0.0,
                average_confidence=0.0,
                override_frequency=0.0,
                learning_rate=0.0,
            )

        history = self.collaboration_history
        agreements = sum(result.human_decision.action == "approve" for result in history)
        agreement_rate = agreements / len(history)
        average_confidence = float(
            np.mean([result.human_decision.confidence for result in history])
        )
        overrides = sum(
            result.human_decision.action in {"override", "modify"} for result in history
        )
        override_frequency = overrides / len(history)

        # A relative difference is reported only after enough records exist for both windows.
        if len(history) > 10:
            early = np.mean(
                [result.performance_outcome["service_level"] for result in history[:5]]
            )
            recent = np.mean(
                [result.performance_outcome["service_level"] for result in history[-5:]]
            )
            learning_rate = float((recent - early) / early) if early else 0.0
        else:
            learning_rate = 0.0

        metrics = TrustMetrics(
            timestamp=datetime.now(),
            agreement_rate=agreement_rate,
            average_confidence=average_confidence,
            override_frequency=override_frequency,
            learning_rate=learning_rate,
        )
        self.trust_metrics.append(metrics)
        return metrics

    def generate_collaboration_report(self) -> Dict[str, Any]:
        """Summarize decision counts and simulated performance for available records."""
        trust_metrics = self.calculate_trust_metrics()
        decision_patterns = {
            action: sum(
                result.human_decision.action == action
                for result in self.collaboration_history
            )
            for action in ("approve", "modify", "override", "reject")
        }

        if self.collaboration_history:
            average_service_level = float(
                np.mean([
                    result.performance_outcome["service_level"]
                    for result in self.collaboration_history
                ])
            )
            average_cost = float(
                np.mean([
                    result.performance_outcome["total_cost"]
                    for result in self.collaboration_history
                ])
            )
        else:
            average_service_level = 0.0
            average_cost = 0.0

        return {
            "total_decisions": len(self.collaboration_history),
            "trust_metrics": trust_metrics,
            "decision_patterns": decision_patterns,
            "performance_metrics": {
                "average_service_level": average_service_level,
                "average_cost": average_cost,
            },
            "learning_insights": [
                insight
                for result in self.collaboration_history
                for insight in result.learning_insights
            ],
        }

In [ ]:
# Initialize the human–AI partnership components.
print("Initializing Human–AI Partnership...")
print("=" * 50)

ai_engine = ExplainableAIEngine(model_version="Centaur-1.0")
human_experts = [
    HumanExpert("Sarah Chen", "senior"),
    HumanExpert("Michael Rodriguez", "senior"),
    HumanExpert("Emily Johnson", "lead"),
]
centaur_partnership = CentaurPartnership(ai_engine, human_experts)

print(f"AI Engine: {ai_engine.model_version}")
print(f"Human Experts: {len(human_experts)}")
for expert in human_experts:
    print(f"  - {expert.name} ({expert.expertise_level})")
print("Partnership initialized successfully.")

In [ ]:
# Process the illustrative scenarios using the same AI–human sequence.
print("\nSimulating Human–AI Collaborative Decision Making...")
print("=" * 60)

test_scenarios = [
    {
        "location": "DC-Phoenix",
        "sku": "SKU-123-Critical",
        "current_inventory": 150,
        "demand_forecast": 80,
        "demand_volatility": 15,
        "supplier_reliability": 0.92,
        "lead_time": 5,
        "current_service_level": 0.94,
        "target_service_level": 0.99,
    },
    {
        "location": "DC-Dallas",
        "sku": "SKU-456-Standard",
        "current_inventory": 200,
        "demand_forecast": 60,
        "demand_volatility": 8,
        "supplier_reliability": 0.88,
        "lead_time": 3,
        "current_service_level": 0.96,
        "target_service_level": 0.95,
    },
    {
        "location": "Warehouse-Chicago",
        "sku": "SKU-789-SlowMoving",
        "current_inventory": 300,
        "demand_forecast": 25,
        "demand_volatility": 5,
        "supplier_reliability": 0.95,
        "lead_time": 7,
        "current_service_level": 0.98,
        "target_service_level": 0.90,
    },
    {
        "location": "DC-Miami",
        "sku": "SKU-101-Seasonal",
        "current_inventory": 120,
        "demand_forecast": 100,
        "demand_volatility": 25,
        "supplier_reliability": 0.78,
        "lead_time": 4,
        "current_service_level": 0.91,
        "target_service_level": 0.95,
    },
]

collaboration_results = []
for i, scenario in enumerate(test_scenarios, start=1):
    print(f"\nScenario {i}: {scenario['sku']} at {scenario['location']}")
    print("-" * 50)
    result = centaur_partnership.process_inventory_decision(scenario)
    collaboration_results.append(result)

    ai_rec = result.ai_recommendation
    print(f"AI Recommendation: {ai_rec.action_type}")
    print(f"  Current Level: {ai_rec.current_level:.0f}")
    print(f"  Recommended Level: {ai_rec.recommended_level:.0f}")
    print(f"  Confidence: {ai_rec.confidence_score:.1%}")
    print(f"  Risk Assessment: {ai_rec.risk_assessment}")
    print(f"  AI Reasoning: {'; '.join(ai_rec.reasoning[:2])}")

    human_decision = result.human_decision
    print(f"\nHuman Decision ({human_decision.human_expert}): {human_decision.action}")
    print(f"  Final Level: {human_decision.final_level:.0f}")
    print(f"  Human Confidence: {human_decision.confidence:.1%}")
    print(f"  Human Reasoning: {human_decision.reasoning}")
    if human_decision.contextual_factors:
        print(f"  Contextual Factors: {', '.join(human_decision.contextual_factors)}")

    outcome = result.performance_outcome
    print("\nIllustrative Simulated Outcome:")
    print(f"  Service Level: {outcome['service_level']:.1%}")
    print(f"  Total Cost: ${outcome['total_cost']:.0f}")
    print(f"  Customer Satisfaction Index: {outcome['customer_satisfaction']:.1f}")

In [ ]:
# Summarize the decisions and simulated outcomes recorded above.
print("\n" + "=" * 60)
print("HUMAN–AI COLLABORATION SUMMARY")
print("=" * 60)

report = centaur_partnership.generate_collaboration_report()
trust = report["trust_metrics"]
patterns = report["decision_patterns"]
performance = report["performance_metrics"]
total_decisions = report["total_decisions"]

print(f"\nTotal Decisions Processed: {total_decisions}")
print("\nDescriptive Collaboration Indicators:")
print("-" * 32)
print(f"Approval Rate (Agreement Proxy): {trust.agreement_rate:.1%}")
print(f"Average Human Confidence: {trust.average_confidence:.1%}")
print(f"Modification/Override Frequency: {trust.override_frequency:.1%}")
print(f"Early-to-Recent Relative Service-Level Change: {trust.learning_rate:.1%}")
print("These indicators describe this demonstration; they do not measure trust directly.")

print("\nDecision Patterns:")
print("-" * 20)
for action, count in patterns.items():
    percentage = count / total_decisions * 100 if total_decisions else 0.0
    print(f"{action.capitalize():<10}: {count:2d} ({percentage:5.1f}%)")

print("\nSimulated Performance Metrics:")
print("-" * 28)
print(f"Average Simulated Service Level: {performance['average_service_level']:.1%}")
print(f"Average Simulated Cost: ${performance['average_cost']:.0f}")

print("\nRecorded Feedback Notes:")
print("-" * 24)
insights = report["learning_insights"]
if insights:
    for insight in insights[:5]:
        print(f"- {insight}")
else:
    print("No feedback notes are available.")

print("\nInterpretation:")
print("-" * 14)
if total_decisions:
    print(
        "Approval and modification rates summarize how the predefined human rules "
        "responded to the AI recommendations."
    )
    print(
        "The simulated service levels and costs are illustrative and do not establish "
        "comparative decision quality."
    )
else:
    print("Process scenarios before interpreting decision or outcome metrics.")

In [ ]:
# Visualize decision counts, descriptive indicators, and simulated outcomes.
if not collaboration_results:
    print("No collaboration records are available to visualize.")
else:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle("Human–AI Partnership: Descriptive Analysis", fontsize=16, fontweight="bold")

    # Decision counts show how often each human response occurred in this sample.
    ax1 = axes[0, 0]
    actions = list(patterns.keys())
    counts = [patterns[action] for action in actions]
    colors = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D"]
    bars = ax1.bar(actions, counts, color=colors, alpha=0.8)
    ax1.set_ylabel("Number of decisions")
    ax1.set_title("Human Decision Counts")
    ax1.grid(True, axis="y", alpha=0.3)
    for bar, count in zip(bars, counts):
        ax1.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            str(count),
            ha="center",
            va="bottom",
        )

    # These are descriptive indicators, not validated psychological trust measures.
    ax2 = axes[0, 1]
    categories = ["Approval proxy", "Confidence", "1 − override frequency"]
    values = [
        trust.agreement_rate,
        trust.average_confidence,
        1 - trust.override_frequency,
    ]
    ax2.bar(categories, values, color=colors[:3], alpha=0.8)
    ax2.set_ylabel("Proportion")
    ax2.set_title("Descriptive Collaboration Indicators")
    ax2.set_ylim(0, 1)
    ax2.tick_params(axis="x", rotation=15)
    ax2.grid(True, axis="y", alpha=0.3)

    # No AI-only or human-only measurements are generated in this notebook.
    ax3 = axes[0, 2]
    service_by_action = {
        action: [
            result.performance_outcome["service_level"]
            for result in collaboration_results
            if result.human_decision.action == action
        ]
        for action in actions
    }
    nonempty_actions = [action for action in actions if service_by_action[action]]
    ax3.boxplot(
        [service_by_action[action] for action in nonempty_actions],
        tick_labels=nonempty_actions,
        patch_artist=True,
    )
    ax3.set_ylabel("Simulated service level")
    ax3.set_title("Outcomes by Human Decision")
    ax3.tick_params(axis="x", rotation=15)
    ax3.grid(True, axis="y", alpha=0.3)

    # Approval is used as a simple agreement proxy; each point is one decision.
    ax4 = axes[1, 0]
    for result in collaboration_results:
        approved = result.human_decision.action == "approve"
        ax4.scatter(
            result.human_decision.confidence,
            int(approved),
            color="green" if approved else "orange",
            alpha=0.8,
            s=80,
            label="Approved" if approved else "Modified or overridden",
        )
    handles, labels = ax4.get_legend_handles_labels()
    unique_labels = dict(zip(labels, handles))
    ax4.legend(unique_labels.values(), unique_labels.keys(), fontsize=8)
    ax4.set_xlabel("Human confidence")
    ax4.set_ylabel("Approval proxy (1 = approve, 0 = other)")
    ax4.set_title("Confidence and Approval")
    ax4.set_ylim(-0.1, 1.1)
    ax4.grid(True, alpha=0.3)

    # The sequence is not a learning curve: outcomes are stochastic and only four cases are shown.
    ax5 = axes[1, 1]
    service_levels = [
        result.performance_outcome["service_level"] for result in collaboration_results
    ]
    ax5.plot(range(1, len(service_levels) + 1), service_levels, "o-", linewidth=2)
    ax5.set_xlabel("Scenario number")
    ax5.set_ylabel("Simulated service level")
    ax5.set_title("Simulated Outcomes by Scenario")
    ax5.grid(True, alpha=0.3)

    # This plot describes the cost/service values produced by the illustrative simulator.
    ax6 = axes[1, 2]
    costs = [result.performance_outcome["total_cost"] for result in collaboration_results]
    scatter = ax6.scatter(
        costs,
        service_levels,
        c=range(1, len(costs) + 1),
        cmap="viridis",
        alpha=0.8,
        s=90,
    )
    colorbar = fig.colorbar(scatter, ax=ax6)
    colorbar.set_label("Scenario number")
    ax6.set_xlabel("Simulated total cost ($)")
    ax6.set_ylabel("Simulated service level")
    ax6.set_title("Simulated Cost–Service Relationship")
    ax6.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

In [ ]:
# Present a concise interpretation without inferring unsupported comparisons.
print("\n" + "=" * 60)
print("HUMAN–AI PARTNERSHIP: FINAL SUMMARY")
print("=" * 60)
print()
print("Recorded Demonstration Results:")
print("-" * 30)
print(f"Decisions processed: {report['total_decisions']}")
print(f"Mean simulated service level: {performance['average_service_level']:.1%}")
print(f"Approval rate (agreement proxy): {trust.agreement_rate:.1%}")
print(f"Mean human confidence: {trust.average_confidence:.1%}")
print()
print("What the workflow demonstrates:")
print("-" * 31)
print("- The AI heuristic exposes its target level, assumptions, and reasoning.")
print("- A human can use predefined contextual rules to approve or adjust a recommendation.")
print("- Decisions and illustrative outcomes can be recorded for later review.")
print("- Summary indicators help describe the decisions in the current sample.")
print()
print("Limits on interpretation:")
print("-" * 25)
print("- Random simulated outcomes are not evidence of real-world performance.")
print("- No AI-only or human-only baseline is evaluated.")
print("- Feedback is logged, but no model parameters or human knowledge are learned here.")
print("- The inventory heuristic does not jointly optimize multiple echelons or costs.")

## Key Insights from the Human–AI Partnership Demonstration

### Complementary roles

The example separates numerical analysis from contextual review. The AI component applies a repeatable heuristic and exposes its inputs; the human planner applies predefined contextual rules. This division illustrates a possible workflow, but the notebook does not establish that the combination is superior to either component alone.

### Explainability

A recommendation is easier to review when its target level, supporting values, and stated reasoning are visible. Explanations make assumptions inspectable; they do not, by themselves, guarantee that the recommendation is accurate or that a planner will trust it.

### Contextual review

Promotions, supplier changes, and capacity constraints can matter to inventory decisions. Here, those factors are encoded as fixed examples. A production process would require reliable, current contextual data and clear accountability for adjustments.

### Feedback and learning

The workflow stores decisions and feedback notes. It does not update the recommendation formula or learn new contextual rules. A genuine learning process would need a specified update rule, suitable outcome data, and evaluation that separates the effect of a decision from other causes.

### Interpreting metrics

Approval rate is used as a simple proxy for agreement, and modification/override frequency summarizes non-approval decisions. Neither is a complete measure of trust or decision quality. The early-to-recent indicator is a relative difference between two windows of simulated outcomes, not a measure of learning. The small scenario set and randomly simulated outcomes are descriptive demonstrations only.

### When this workflow may be useful

Human review can be valuable when recommendations have material consequences, contextual information is important, or organizational policy requires human accountability. Applying this workflow to operational planning requires validating the inventory model, outcome measures, data quality, and decision responsibilities.